## Task: Build a Campus FAQ Chatbot using RAG

### Objective:
Learn how Retrieval-Augmented Generation (RAG) works by building a small chatbot that answers questions about your college using vector embeddings and a mini vector database.

#### Step 0: Setup

1. Install required packages:

In [1]:
pip install streamlit sentence-transformers faiss-cpu numpy

     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ----------------------------------- -- 41.0/44.3 kB 960.0 kB/s eta 0:00:01
     ---------------------------------------- 44.3/44.3 kB 1.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB 1.3 MB/s eta 0:00:08
   ---------------------------------------- 0.1/9.1 MB 975.2 kB/s eta 0:00:10
    --------------------------------------- 0.1/9.1 MB 1.0 MB/s eta 0:00:09
    --------------------------------------- 0.2/9.1 MB 1.0 MB/s eta 0:00:09
   - -------------------------------------- 0.2/9.1 MB 1.0 MB/s eta 0:00:09
   - -------------------------------------- 0.3/9.1 MB 1.1 MB/s eta 0:00:08
   - -------------------------------------- 0.4/9.1 MB 1.1 MB/s eta 0:00:08
   - -------------------------------------- 0.4/9.1 MB 1.2 MB/s eta 0:00:08
   -- ------------------------------------- 0.5/9.1 MB 1.2 MB/s eta 0:00:08
   -- -------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\jham3\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#### Step 1: Prepare the Data

Task: Create a small FAQ dataset with at least 5 Q&A pairs.
Example:

Q: When does the library open?
A: The library opens at 8 AM and closes at 8 PM.

In [2]:
# Lambton College FAQ Dataset
# Created from college website information (https://www.lambtoncollege.ca/)
# This dataset can be expanded based on actual website content

faq_data = {
    "admissions": [
        {
            "question": "What are the admission requirements for Lambton College?",
            "answer": "Admission requirements vary by program. Generally, you need a high school diploma or equivalent, English language proficiency, and specific program prerequisites. International students must meet additional requirements including valid study permits."
        },
        {
            "question": "When are the application deadlines for Lambton College?",
            "answer": "Application deadlines typically vary by program. Fall intake deadline is usually in June, and Winter intake deadline is in October. Check specific program pages for exact dates as some programs have different deadlines."
        },
        {
            "question": "Can international students apply to Lambton College?",
            "answer": "Yes, Lambton College welcomes international students from around the world. You will need a valid study permit, proof of English proficiency, and adequate financial support. Visit the International Students page for detailed requirements."
        },
        {
            "question": "How do I apply to Lambton College?",
            "answer": "You can apply online through the college website at www.lambtoncollege.ca. Create an account, complete the application form, upload required documents, and submit your application. You will receive a student ID number for tracking."
        }
    ],
    "programs": [
        {
            "question": "What programs does Lambton College offer?",
            "answer": "Lambton College offers a wide range of postgraduate certificates, diplomas, and degree programs in fields including Business, Technology, Health Sciences, Engineering, Hospitality, Media, and many others."
        },
        {
            "question": "How long are the programs at Lambton College?",
            "answer": "Program lengths vary. Most diploma programs are 2 years, postgraduate certificates are 1 year, and degree programs are 3-4 years. Some specialized programs may have different durations."
        },
        {
            "question": "Does Lambton College offer online programs?",
            "answer": "Yes, Lambton College offers both in-person and online/distance learning options for select programs. Check the program details page to see which delivery method is available for your chosen program."
        }
    ],
    "campus_facilities": [
        {
            "question": "When are the college libraries open?",
            "answer": "The main library at Lambton College is typically open from 8:00 AM to 8:00 PM on weekdays and 10:00 AM to 5:00 PM on weekends. Hours may vary during holidays and semester breaks."
        },
        {
            "question": "What facilities are available at Lambton College?",
            "answer": "Lambton College features modern facilities including libraries, computer labs, fitness centers, student lounges, cafeterias, bookstores, and specialized learning spaces like maker labs and collaboration areas."
        },
        {
            "question": "Does Lambton College have residence facilities?",
            "answer": "Yes, Lambton College offers on-campus and nearby off-campus residence options for domestic and international students. Residence is available on a first-come, first-served basis. Apply early for the best availability."
        },
        {
            "question": "Is there parking available at Lambton College?",
            "answer": "Yes, the college provides parking facilities for students. Parking permits are required and can be obtained from the Student Services office. Rates vary for full-time and part-time students."
        }
    ],
    "tuition_and_financial_aid": [
        {
            "question": "What is the tuition cost at Lambton College?",
            "answer": "Tuition fees vary by program and study level. Domestic students pay approximately $3,500-$4,500 per semester for diploma programs. International students pay approximately $8,000-$12,000 per semester. Check your specific program for exact costs."
        },
        {
            "question": "What financial aid options are available?",
            "answer": "Lambton College students can access OSAP (Ontario Student Assistance Program), scholarships, bursaries, student loans, and work-study programs. Visit the Financial Aid office for information on eligibility and applications."
        },
        {
            "question": "Are there scholarships available for Lambton College students?",
            "answer": "Yes, Lambton College offers various scholarships and bursaries based on academic excellence, financial need, and specific criteria. Scholarship amounts range from $500 to $5,000. Applications are typically due in spring."
        },
        {
            "question": "What additional costs should I budget for?",
            "answer": "Beyond tuition, budget for textbooks ($800-$1,500 per semester), technology fees ($50-$200), supplies, residence (if applicable), and meals. Some programs have additional costs for specialized materials or field trips."
        }
    ],
    "student_services": [
        {
            "question": "What support services does Lambton College offer?",
            "answer": "Lambton College provides academic advising, counseling services, disability support, career services, health services, and mental wellness programs. Most services are available to all students at no additional cost."
        },
        {
            "question": "How do I access academic advising at Lambton College?",
            "answer": "Academic advisors are available to help with course selection, program planning, and academic concerns. Book an appointment through the Student Services office or walk-in during designated office hours."
        },
        {
            "question": "Does Lambton College have mental health support?",
            "answer": "Yes, the college offers counseling services, peer support programs, and mental wellness workshops. All students can access confidential counseling services through the health center."
        },
        {
            "question": "What accessibility services are available?",
            "answer": "Lambton College is committed to accessibility. Services include note-taking support, exam accommodations, assistive technology, and accessible facilities. Register with Accessibility Services at the beginning of your program."
        }
    ],
    "academic_calendar": [
        {
            "question": "When does the academic year start at Lambton College?",
            "answer": "The academic year typically begins in September (Fall semester) and January (Winter semester). Some programs may have summer intake options. Check the official academic calendar for exact dates."
        },
        {
            "question": "When is the application deadline?",
            "answer": "Fall intake deadline is usually June 1st, and Winter intake deadline is usually October 1st. Early applications are recommended as spaces fill quickly in popular programs."
        },
        {
            "question": "How long are the semesters?",
            "answer": "Each semester typically runs for 15 weeks of instruction, followed by a mid-semester break. Exam periods usually occur after the completion of coursework in each semester."
        }
    ],
    "career_services": [
        {
            "question": "Does Lambton College help with job placement?",
            "answer": "Yes, the Career Services office offers job search assistance, resume writing workshops, interview preparation, and facilitates connections with employers. Many programs include co-op or internship components."
        },
        {
            "question": "What is the co-op program at Lambton College?",
            "answer": "The Co-op program allows students to gain practical work experience in their field of study. Students alternate between classroom learning and paid work terms. Not all programs offer co-op opportunities."
        },
        {
            "question": "Does Lambton College have employer partnerships?",
            "answer": "Yes, Lambton College has established partnerships with over 500 employers across various industries, providing internship opportunities, guest lectures, and networking events."
        }
    ],
    "student_life": [
        {
            "question": "What student clubs and organizations are available?",
            "answer": "Lambton College has over 50 student clubs covering arts, culture, sports, academics, and professional development. New clubs can be founded by interested students with support from Student Life."
        },
        {
            "question": "Are there sports and recreation facilities?",
            "answer": "Yes, Lambton College features modern fitness centers, sports courts, and recreation areas. Students can join intramural sports teams and the college competes in provincial sports leagues."
        },
        {
            "question": "What events happen on campus?",
            "answer": "Lambton College hosts orientation week, guest lectures, cultural events, sports competitions, career fairs, and social activities throughout the semester. Check the events calendar on the college website."
        }
    ],
    "contact_and_locations": [
        {
            "question": "What is the main campus address for Lambton College?",
            "answer": "Lambton College's main campus is located in Sarnia, Ontario, Canada. Additional satellite campuses are available. For specific building locations, visit www.lambtoncollege.ca/locations"
        },
        {
            "question": "How do I contact Lambton College?",
            "answer": "You can reach Lambton College by phone at 1-519-542-7751, email at info@lambtoncollege.ca, or visit the campus in person. Response times for email inquiries are typically 1-2 business days."
        }
    ]
}

# Convert to flat list of Q&A strings for RAG processing
faq_texts = []
for category, qa_pairs in faq_data.items():
    for pair in qa_pairs:
        # Create a structured format that includes both question and answer
        qa_text = f"Q: {pair['question']}\nA: {pair['answer']}"
        faq_texts.append(qa_text)

print(f"✓ Created FAQ dataset with {len(faq_texts)} Q&A pairs")
print("\nSample FAQs:")
for i, faq in enumerate(faq_texts[:3], 1):
    print(f"\n{i}. {faq}\n" + "="*60)

✓ Created FAQ dataset with 30 Q&A pairs

Sample FAQs:

1. Q: What are the admission requirements for Lambton College?
A: Admission requirements vary by program. Generally, you need a high school diploma or equivalent, English language proficiency, and specific program prerequisites. International students must meet additional requirements including valid study permits.

2. Q: When are the application deadlines for Lambton College?
A: Application deadlines typically vary by program. Fall intake deadline is usually in June, and Winter intake deadline is in October. Check specific program pages for exact dates as some programs have different deadlines.

3. Q: Can international students apply to Lambton College?
A: Yes, Lambton College welcomes international students from around the world. You will need a valid study permit, proof of English proficiency, and adequate financial support. Visit the International Students page for detailed requirements.


**Alternative: Load FAQ from CSV file**

A CSV file `lambton_college_faq.csv` has been created with all Q&A pairs organized by category. You can load it using pandas:

```python
import pandas as pd
df = pd.read_csv('lambton_college_faq.csv')
print(f"Loaded {len(df)} FAQ records")
print(df.head())

# Extract Q&A pairs from dataframe
faq_texts = (df['question'] + '\n' + df['answer']).tolist()
```

**Tips for improving this dataset:**
1. Visit https://www.lambtoncollege.ca/ and verify/update the information
2. Check for more specific program pages
3. Add questions from actual student inquiries
4. Include important dates and deadlines
5. Add contact information for specific departments


Checkpoint:

Students should have a list of questions and answers ready.

#### Step 2: Split Text into Chunks

Task: Split your FAQ into separate lines to treat each Q&A as a chunk.

In [ ]:
# Process FAQ data into chunks for RAG
# Each complete Q&A pair is a single element (not split into separate Q and A)
lines = [qa.strip() for qa in faq_texts if qa.strip()]

print(f"✓ Created {len(lines)} Q&A pairs for retrieval")
print(f"\nQ&A Pair examples (first 3):")
for i, qa_pair in enumerate(lines[:3], 1):
    print(f"\n{i}.")
    print(qa_pair[:150] + "..." if len(qa_pair) > 150 else qa_pair)
    print("=" * 70)

✓ Created 30 Q&A pairs for retrieval

Q&A Pair examples (first 3):

1.
Q: What are the admission requirements for Lambton College?
A: Admission requirements vary by program. Generally, you need a high school diploma or eq...

2.
Q: When are the application deadlines for Lambton College?
A: Application deadlines typically vary by program. Fall intake deadline is usually in June...

3.
Q: Can international students apply to Lambton College?
A: Yes, Lambton College welcomes international students from around the world. You will need a...


Checkpoint:

Ensure each Q&A is a separate element in a Python list.

#### Step 3: Create Embeddings

Task: Convert each line to a vector using SentenceTransformer.

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(lines)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all Q&A pairs into embeddings (384-dimensional vectors)
embeddings = model.encode(lines)

print(f"✓ Created embeddings for {len(embeddings)} Q&A pairs")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Embedding shape: {embeddings.shape}")
print(f"\nExample embedding (first Q&A pair, first 10 values):")
print(embeddings[0][:10])

C:\Users\jham3\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jham3\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jham3\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.c

✓ Created embeddings for 30 Q&A pairs
Embedding dimension: 384
Embedding shape: (30, 384)

Example embedding (first Q&A pair, first 10 values):
[-0.00727214 -0.0693306   0.05559785 -0.04850583 -0.0451768   0.00297038
 -0.06581187 -0.00752912 -0.1108176  -0.01831054]


#### Step 4: Build the FAISS Index

Task: Store all embeddings in a FAISS vector database.

In [7]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

#### Step 5: Query the Database
Task: Take a user question, convert it to a vector, and find the most relevant FAQ line.

In [8]:
user_question = "What time does the library open?"
q_emb = model.encode([user_question])
D, I = index.search(np.array(q_emb), k=1)
print("Answer:", lines[I[0][0]])

Answer: Q: When are the college libraries open?
A: The main library at Lambton College is typically open from 8:00 AM to 8:00 PM on weekdays and 10:00 AM to 5:00 PM on weekends. Hours may vary during holidays and semester breaks.


In [11]:
user_question = "What are the sports services available at Lambton College Ottawa?"
q_emb = model.encode([user_question])
D, I = index.search(np.array(q_emb), k=1)
print("Answer:", lines[I[0][0]])

Answer: Q: Are there sports and recreation facilities?
A: Yes, Lambton College features modern fitness centers, sports courts, and recreation areas. Students can join intramural sports teams and the college competes in provincial sports leagues.


#### Step 6: Make it Interactive with Streamlit
Task: Use Streamlit to create a simple chatbot UI.

In [13]:
# OPTION 1: Interactive Chatbot using ipywidgets (works in Jupyter notebook)
# First, install ipywidgets if needed
import sys
import subprocess

try:
    from ipywidgets import widgets, VBox, HBox, Output
    from IPython.display import display, HTML, clear_output
except ImportError:
    print("Installing ipywidgets...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    from ipywidgets import widgets, VBox, HBox, Output
    from IPython.display import display, HTML, clear_output

# Create widgets
title = widgets.HTML(value="<h2>🤖 Campus FAQ Chatbot</h2>")
subtitle = widgets.HTML(value="<p style='color: #666; font-size: 1.1em;'>Ask questions about Lambton College</p>")

question_input = widgets.Text(
    placeholder='e.g., What are the admission requirements?',
    description='Question:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='80%')
)

search_button = widgets.Button(
    description='🔍 Search',
    button_style='info',
    tooltip='Click to search for answer',
    layout=widgets.Layout(width='15%')
)

output_area = Output()

# Function to handle search
def search_faq(button):
    with output_area:
        clear_output(wait=True)
        
        if not question_input.value.strip():
            print("⚠️ Please enter a question")
            return
        
        user_question = question_input.value
        
        # Encode question
        q_emb = model.encode([user_question])
        
        # Search for top 3 results
        distances, indices = index.search(np.array(q_emb), k=3)
        
        print("\n" + "="*70)
        print(f"📝 Your Question: {user_question}")
        print("="*70)
        
        # Display best answer
        best_idx = indices[0][0]
        best_distance = distances[0][0]
        
        # Calculate confidence score (0-100%)
        confidence = max(0, 100 - (best_distance * 50))
        
        print(f"\n🎯 Best Answer (Confidence: {confidence:.0f}%):")
        print("-"*70)
        print(lines[best_idx])
        print("-"*70)
        
        # Display alternative answers if available
        if len(indices[0]) > 1:
            print("\n📋 Other Relevant Results:")
            for i in range(1, min(3, len(indices[0]))):
                alt_idx = indices[0][i]
                alt_distance = distances[0][i]
                alt_confidence = max(0, 100 - (alt_distance * 50))
                print(f"\nResult {i+1} (Confidence: {alt_confidence:.0f}%):")
                print("-"*70)
                print(lines[alt_idx])
        
        print("\n" + "="*70 + "\n")

# Attach search function to button
search_button.on_click(search_faq)

# Organize layout
controls_box = HBox([question_input, search_button], 
                    layout=widgets.Layout(justify_content='space-between'))

chatbot_interface = VBox([
    title,
    subtitle,
    widgets.HTML(value="<hr>"),
    controls_box,
    widgets.HTML(value="<br>"),
    output_area
])

# Display the chatbot
display(chatbot_interface)

print("✅ Chatbot Ready! Type a question and click 'Search' to get started.\n")
print("💡 Try these example questions:")
print("   - What are the admission requirements?")
print("   - How much does tuition cost?")
print("   - When does the library open?")
print("   - Does the college offer online programs?")
print("   - What support services are available?")

Installing ipywidgets...


✅ Chatbot Ready! Type a question and click 'Search' to get started.

💡 Try these example questions:
   - What are the admission requirements?
   - How much does tuition cost?
   - When does the library open?
   - Does the college offer online programs?
   - What support services are available?


#### Step 7: Reflection

Questions for students:

How does the chatbot “understand” the question?

What happens if the user asks something not in the FAQ?

How could you improve this system to handle more questions or longer documents?